# 01 — GR-source ablation: what if the GR input is something else?

**No training.** The same `GainPriorDiffSSLLSTM` checkpoint is streamed with the GR
input swapped between sources, all scored with the canonical 9-column metric engine
on the full validation and test splits. This is the *cascade evaluation with error
budget* — the honest end-to-end number that fixes the §5.3 oracle-leakage caveat of
`06_output/MODEL_GAIN_PRIOR.md` — plus the GR-convention variants that probe how much
the model's corrections are tied to the exact label definition.

| variant | GR source | network | what it isolates |
|---|---|---|---|
| `oracle` | exported curve (uses wet) | gain-prior | upper bound (leaks target envelope) |
| `detector` | 05 `DetectorGRLSTM` (dry + knobs only) | gain-prior | **deployable end-to-end cascade** |
| `amp_match_oracle` | exported curve | *bypassed* | the raw multiply prior (no NN) |
| `amp_match_detector` | 05 predictor | *bypassed* | stage 1 alone (gain computer + multiply) |
| `const_mean` | whole-song mean GR (scalar) | gain-prior | correct average depth, zero dynamics |
| `const_0db` | 0 dB everywhere | gain-prior | no compression information at all |
| `oracle_w256` | GR recomputed from dry/wet, RMS win 256 (5.8 ms) | gain-prior | label-convention sensitivity: a *faster* prior |
| `oracle_w4096` | GR recomputed, RMS win 4096 (93 ms) | gain-prior | a *slower*, more smeared prior — must Δg re-sharpen more? |

Error-budget reading (last cells): `detector − oracle` = cost of predicting the gain
trajectory; `oracle − amp_match_oracle` = what the network adds on top of the multiply;
`amp_match_detector − detector` = what the network absorbs of the upstream GR error.
`const_*` rows calibrate how much of the performance is carried by the GR input at all.

> Run from `07_experiments/` with the repo venv (`uv sync`). Full splits ≈ 30–60 min
> on CPU; set `MAX_PAIRS` for a smoke run. Results → `eval_output/*.csv`.

In [ ]:
# -- 0. Setup: models, split, pairs ------------------------------------
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from exp_common import (
    DATA_ROOT, DEVICE, GR_DB_MAX, GR_DB_MIN, METRIC_COLS, SR,
    aggregate_rows, amplitude_match, ensure_eval_out, gain_reduction_db,
    load_detector, load_gain_prior, load_pair, load_split, pairs_from_keys,
    params_for, score_signal, stream_detector_gr, stream_gain_prior,
)

# None -> newest run of the type; keep the documented runs for reproducibility.
SELECTED_GAIN_PRIOR_RUN = "gain_prior_20260702_085618_diffssl_lstm32_gain_prior"
SELECTED_DETECTOR_RUN = "lstm_gr_20260702_174039_lstm_detector_gr"

MODEL, HP, RUN_DIR = load_gain_prior(SELECTED_GAIN_PRIOR_RUN)
DET, DET_HP, DET_DIR = load_detector(SELECTED_DETECTOR_RUN)
SPLIT = load_split(RUN_DIR)
VAL_PAIRS = pairs_from_keys(SPLIT.val_pair_keys)
TEST_PAIRS = pairs_from_keys(SPLIT.test_pair_keys)
OUT = ensure_eval_out()

MAX_PAIRS: int | None = None   # small int for a smoke run

print(f"val pairs: {len(VAL_PAIRS)} ({SPLIT.val_songs} x all settings)")
print(f"test pairs: {len(TEST_PAIRS)} ({SPLIT.test_songs} x {SPLIT.test_settings})")

In [ ]:
# -- 1. GR sources ------------------------------------------------------
# Each source returns (gr [1, T] dB, bypass_network). The oracle_w* variants
# recompute GR from dry/wet with a DIFFERENT causal RMS window than the label's
# 1024 - same information, different smoothing convention.

VARIANTS = ["oracle", "detector", "amp_match_oracle", "amp_match_detector",
            "const_mean", "const_0db", "oracle_w256", "oracle_w4096"]


def gr_sources(song, setting, dry, wet, gr_oracle):
    T = dry.shape[-1]
    gr_det = stream_detector_gr(DET, dry, params_for(setting), sample_len=T)
    gr_w256 = gain_reduction_db(dry, wet, 256).clamp(GR_DB_MIN, GR_DB_MAX)
    gr_w4096 = gain_reduction_db(dry, wet, 4096).clamp(GR_DB_MIN, GR_DB_MAX)
    return {
        "oracle": (gr_oracle, False),
        "detector": (gr_det, False),
        "amp_match_oracle": (gr_oracle, True),
        "amp_match_detector": (gr_det, True),
        "const_mean": (torch.full_like(gr_oracle, float(gr_oracle.mean())), False),
        "const_0db": (torch.zeros_like(gr_oracle), False),
        "oracle_w256": (gr_w256, False),
        "oracle_w4096": (gr_w4096, False),
    }

In [ ]:
# -- 2. Run: stream every (pair x variant), score with the 9-column engine --
# Whole songs, stateful streaming (reset per pair, block-aligned chunks), the
# same protocol as the 05/06 eval notebooks.

def run_split(pairs, split_name):
    pairs = pairs if MAX_PAIRS is None else pairs[:MAX_PAIRS]
    chunk_rows = {v: [] for v in VARIANTS}
    pair_rows = []
    for i, (song, setting) in enumerate(pairs, start=1):
        dry, wet, gr_oracle = load_pair(setting, song)
        sources = gr_sources(song, setting, dry, wet, gr_oracle)
        p = params_for(setting)
        for variant in VARIANTS:
            gr_v, bypass = sources[variant]
            n = min(dry.shape[-1], gr_v.shape[-1])
            if bypass:
                pred = amplitude_match(dry[..., :n], gr_v[..., :n])
            else:
                pred = stream_gain_prior(MODEL, dry[..., :n], gr_v[..., :n], p)
            rows = score_signal(dry, pred, wet[..., :n])
            chunk_rows[variant] += rows
            pair_rows.append({"Split": split_name, "Variant": variant,
                              "Song": song, "Setting": setting,
                              "Frames": sum(r["Frames"] for r in rows),
                              **aggregate_rows(rows)})
            print(f"[{split_name} {i}/{len(pairs)}] {song[:16]:16s} {setting:45s} "
                  f"{variant:20s} GR MAE {pair_rows[-1]['GR MAE (dB)']:.3f} dB")
        del sources
        gc.collect()
    split_df = pd.DataFrame([
        {"Split": split_name, "Variant": v,
         "Duration (s)": sum(r["Frames"] for r in chunk_rows[v]) / SR,
         **aggregate_rows(chunk_rows[v])}
        for v in VARIANTS])
    return split_df, pd.DataFrame(pair_rows)


val_df, val_pair_df = run_split(VAL_PAIRS, "val")
val_df

In [ ]:
# -- 3. Test split (held-out songs x lowest-threshold settings) --------
test_df, test_pair_df = run_split(TEST_PAIRS, "test")
test_df

In [ ]:
# -- 4. Save + headline tables -----------------------------------------
all_df = pd.concat([val_df, test_df], ignore_index=True)
all_pair_df = pd.concat([val_pair_df, test_pair_df], ignore_index=True)
all_df.to_csv(OUT / "01_gr_source_ablation_splits.csv", index=False)
all_pair_df.to_csv(OUT / "01_gr_source_ablation_pairs.csv", index=False)
print(f"Saved -> {OUT}/01_gr_source_ablation_(splits|pairs).csv\n")

for split_name, df in (("VALIDATION", val_df), ("TEST", test_df)):
    print(f"== {split_name} ==")
    display(df.set_index("Variant")[METRIC_COLS].round(4))

In [ ]:
# -- 5. Error budget ----------------------------------------------------
# The cascade decomposition the two-stage design promises: additive, separable
# error sources read directly off the variant table.

def row(df, v):
    return df.set_index("Variant").loc[v]

for split_name, df in (("val", val_df), ("test", test_df)):
    orc, det = row(df, "oracle"), row(df, "detector")
    am_o, am_d = row(df, "amp_match_oracle"), row(df, "amp_match_detector")
    print(f"== {split_name} ==")
    for m in ("GR MAE (dB)", "MR-STFT", "ESR (A-wt)", "M_NRMSE"):
        print(f"  {m:12s} oracle {orc[m]:7.4f} | cascade {det[m]:7.4f} "
              f"(cascade cost {det[m] - orc[m]:+7.4f}) | "
              f"prior-only {am_o[m]:7.4f} (NN gain, oracle {am_o[m] - orc[m]:+7.4f}) | "
              f"stage1-only {am_d[m]:7.4f} (NN gain, predicted {am_d[m] - det[m]:+7.4f})")
    print()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, m in zip(axes, ("GR MAE (dB)", "MR-STFT")):
    w = 0.38
    xs = np.arange(len(VARIANTS))
    ax.bar(xs - w / 2, [row(val_df, v)[m] for v in VARIANTS], w, label="val")
    ax.bar(xs + w / 2, [row(test_df, v)[m] for v in VARIANTS], w, label="test")
    ax.set_xticks(xs, VARIANTS, rotation=35, ha="right", fontsize=8)
    ax.set_ylabel(m); ax.grid(alpha=0.3, axis="y"); ax.legend()
fig.suptitle("GR-source ablation - same checkpoint, different conditioning signal")
fig.tight_layout()
fig.savefig(OUT / "01_gr_source_ablation_bars.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# -- 6. GR-curve overlay: what the sources actually look like ----------
song, setting = VAL_PAIRS[len(VAL_PAIRS) // 2]
a, b = int(30 * SR), int(40 * SR)
dry, wet, gr_oracle = load_pair(setting, song, start_sec=0.0)
sources = gr_sources(song, setting, dry, wet, gr_oracle)

t = np.arange(a, b) / SR
fig, ax = plt.subplots(figsize=(13, 4))
for v, style in (("oracle", dict(color="#1f77b4", lw=1.4)),
                 ("detector", dict(color="#d62728", lw=1.0, alpha=0.9)),
                 ("oracle_w256", dict(color="#2ca02c", lw=0.8, alpha=0.7)),
                 ("oracle_w4096", dict(color="#9467bd", lw=1.0, alpha=0.8))):
    ax.plot(t, sources[v][0][0, a:b].numpy(), label=v, **style)
ax.set_xlabel("time (s)"); ax.set_ylabel("GR (dB)"); ax.grid(alpha=0.3)
ax.legend(); ax.set_title(f"{song} | {setting} - GR sources, 10 s window")
fig.tight_layout()
fig.savefig(OUT / "01_gr_sources_overlay.png", dpi=150, bbox_inches="tight")
plt.show()

## Reading the results

- **`detector` vs `oracle`** is the thesis's first leakage-free end-to-end number.
  If the degradation is ≈ the first-order bound (0.268 dB upstream MAE → ≈ 3 % RMS
  envelope error), the interface composes as designed; a blow-up would indicate
  distribution mismatch between oracle-GR training and predicted-GR inference.
- **`amp_match_detector` vs `detector`**: if the full cascade beats the bare
  stage-1 multiply, the gain stage's Δg is absorbing part of the *predictor's*
  error, not just the label's RMS smear — evidence the correction head generalises
  across GR sources.
- **`const_mean` / `const_0db`** calibrate the GR input's information content: how
  far does knob conditioning + dry signal alone carry the model when the dynamic
  conditioning signal is uninformative?
- **`oracle_w256` / `oracle_w4096`** probe convention sensitivity. Training GR used
  a 23 ms window; if `oracle_w256` scores close to `oracle`, the model tolerates a
  faster (less smeared) prior — relevant because the 05 predictor's effective
  smoothing (learned τ ≈ 15–75 ms) differs from the label's.